In [ ]:
# Blood Sugar Predictor (Rule-Based)

# Function to predict blood sugar status
def predict_blood_sugar(FBS, RBS, HbA1c):
    if FBS >= 126 or RBS >= 200 or HbA1c >= 6.5:
        return "Diabetes"
    elif 100 <= FBS <= 125 or 141 <= RBS <= 199 or 5.7 <= HbA1c <= 6.4:
        return "Pre-Diabetes"
    else:
        return "Normal"

# Example patient data
patient_1 = {"FBS": 110, "RBS": 130, "HbA1c": 5.9}
patient_2 = {"FBS": 130, "RBS": 210, "HbA1c": 7.0}
patient_3 = {"FBS": 90, "RBS": 120, "HbA1c": 5.5}

# Predicting
print("Patient 1:", predict_blood_sugar(**patient_1))
print("Patient 2:", predict_blood_sugar(**patient_2))
print("Patient 3:", predict_blood_sugar(**patient_3))

Patient 1: Pre-Diabetes
Patient 2: Diabetes
Patient 3: Normal


In [ ]:
def predict_blood_sugar_extended(FBS, RBS, HbA1c, Age, BMI, Systolic_BP, Diastolic_BP, Exercise, Family_History, Smoking):
    result = predict_blood_sugar(FBS, RBS, HbA1c)

    risk_factor = 0
    # Age risk
    if Age > 45:
        risk_factor += 1
    # BMI risk
    if BMI >= 25:
        risk_factor += 1
    # BP risk
    if Systolic_BP >= 140 or Diastolic_BP >= 90:
        risk_factor += 1
    # Lifestyle
    risk_factor += Exercise==0
    risk_factor += Family_History
    risk_factor += Smoking

    return result, risk_factor

# Example
patient = {"FBS":110, "RBS":150, "HbA1c":5.9, "Age":50, "BMI":28, "Systolic_BP":135, "Diastolic_BP":85, "Exercise":0, "Family_History":1, "Smoking":0}
result, risk = predict_blood_sugar_extended(**patient)
print("Result:", result)
print("Risk Factor:", risk)

Result: Pre-Diabetes
Risk Factor: 4


In [ ]:
# Step 1: Install required libraries
# !pip install streamlit pyngrok qrcode[pil] pandas

In [ ]:
# Replace YOUR_AUTHTOKEN_HERE with your ngrok token
# !ngrok authtoken 318B9FaF6n3uMGKgPpisPTGg6fD_2yLvC69LLhVWTNXrANEwH

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# Step 1: Install required libraries
# !pip install gradio pandas qrcode[pil]

In [ ]:
# Install dependencies (run once)
# pip install gradio pandas qrcode[pil]

import gradio as gr
import pandas as pd
import qrcode
from io import BytesIO
import base64

# Blood sugar prediction functions
def predict_blood_sugar(FBS,RBS,HbA1c):
    if FBS>=126 or RBS>=200 or HbA1c>=6.5:
        return "Diabetes"
    elif 100<=FBS<=125 or 141<=RBS<=199 or 5.7<=HbA1c<=6.4:
        return "Pre-Diabetes"
    else:
        return "Normal"

def predict_risk(FBS,RBS,HbA1c,Age,BMI,Systolic_BP,Diastolic_BP,Exercise,Family_History,Smoking):
    result = predict_blood_sugar(FBS,RBS,HbA1c)
    risk = 0
    if Age>45: risk+=1
    if BMI>=25: risk+=1
    if Systolic_BP>=140 or Diastolic_BP>=90: risk+=1
    risk += Exercise + Family_History + Smoking
    return result, risk

# Process CSV safely
def process_csv(file):
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()  # clean column names

    # Fill missing numeric values
    numeric_cols = df.select_dtypes(include=['float64','int64','int32']).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)

    # Convert Yes/No to 0/1
    for col in ['Exercise','Family_History','Smoking']:
        if col in df.columns:
            df[col] = df[col].map({'Yes':0,'No':1}).fillna(0)
        else:
            df[col] = 0

    # Predict for each row
    results = []
    for _, row in df.iterrows():
        result, risk = predict_risk(
            row.get('FBS',90),
            row.get('RBS',120),
            row.get('HbA1c',5.5),
            row.get('Age',30),
            row.get('BMI',22),
            row.get('Systolic_BP',120),
            row.get('Diastolic_BP',80),
            row.get('Exercise',0),
            row.get('Family_History',0),
            row.get('Smoking',0)
        )

        # Generate QR code as string (base64)
        phone = "+923345534869"
        msg = "I want to order NOORÉ Skin Cream"
        url = f"https://wa.me/{phone[1:]}?text={msg.replace(' ','%20')}"
        qr = qrcode.QRCode(box_size=4)
        qr.add_data(url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = BytesIO()
        img.save(buf, format="PNG")
        qr_base64 = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

        results.append({
            'Blood_Sugar_Status': result,
            'Risk_Factor_Score': risk,
            'QR_Code_Base64': qr_base64
        })

    result_df = pd.DataFrame(results)
    final_df = pd.concat([df.reset_index(drop=True), result_df], axis=1)
    final_df.to_csv("blood_sugar_results.csv", index=False)
    return final_df

# Gradio Interface
iface = gr.Interface(
    fn=process_csv,
    inputs=gr.File(label="Upload CSV File"),
    outputs=gr.Dataframe(label="Predicted Results (CSV will be saved as blood_sugar_results.csv)"),
    title="🩸 Safe Diabetes + Blood Sugar Analysis Tool",
    description="Upload CSV with patient data. Columns: FBS,RBS,HbA1c,Age,BMI,Systolic_BP,Diastolic_BP,Exercise,Family_History,Smoking"
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ad6de1175af963849.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd

# Sample data
data = {
    'FBS':[110,90,130],
    'RBS':[150,120,210],
    'HbA1c':[5.9,5.5,7.0],
    'Age':[50,30,60],
    'BMI':[28,22,30],
    'Systolic_BP':[135,120,145],
    'Diastolic_BP':[85,80,95],
    'Exercise':['Yes','Yes','No'],
    'Family_History':['Yes','No','Yes'],
    'Smoking':['No','No','Yes']
}

df = pd.DataFrame(data)
df.to_csv("sample_blood_sugar.csv", index=False)
print("Sample CSV created as sample_blood_sugar.csv")

Sample CSV created as sample_blood_sugar.csv


In [ ]:
import base64

base64_str = "iVBORw0KGgoAAAANSUhEUgAAALQAAAC0AQAAAAAVtjufAAABlElEQVR4nO2XQYolSQxD..."  # aap ka string without data:image/png;base64,
img_bytes = base64.b64decode(base64_str)

with open("qr_code.png", "wb") as f:
    f.write(img_bytes)

print("QR code image saved as qr_code.png")


QR code image saved as qr_code.png


In [ ]:
df.to_csv("blood_sugar_results.csv", index=False)


In [ ]:
with open("qr_code.png", "wb") as f:
    f.write(img_bytes)


In [ ]:
!ls


app.py			 qr_code.png		 sample_data
blood_sugar_results.csv  sample_blood_sugar.csv


In [ ]:
!pwd


/content


In [ ]:
from google.colab import files
files.download("blood_sugar_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("qr_code.png")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# pip install streamlit

In [ ]:
# Step 1: Install required libraries
# !pip install gradio pandas qrcode[pil]

# Step 2: Import libraries
import gradio as gr
import qrcode
from io import BytesIO
import pandas as pd

# Step 3: Prediction functions
def predict_blood_sugar(FBS,RBS,HbA1c):
    if FBS>=126 or RBS>=200 or HbA1c>=6.5:
        return "Diabetes"
    elif 100<=FBS<=125 or 141<=RBS<=199 or 5.7<=HbA1c<=6.4:
        return "Pre-Diabetes"
    else:
        return "Normal"

def predict_extended(FBS,RBS,HbA1c,Age,BMI,Systolic_BP,Diastolic_BP,Exercise,Family_History,Smoking):
    result = predict_blood_sugar(FBS,RBS,HbA1c)
    risk = 0
    if Age>45: risk+=1
    if BMI>=25: risk+=1
    if Systolic_BP>=140 or Diastolic_BP>=90: risk+=1
    risk += Exercise + Family_History + Smoking
    return result, risk

# Step 4: Batch prediction function
def process_csv(file):
    df = pd.read_csv(file)

    # Fill missing numeric values with safe defaults
    df['FBS'] = df['FBS'].fillna(90)
    df['RBS'] = df['RBS'].fillna(120)
    df['HbA1c'] = df['HbA1c'].fillna(5.5)
    df['Age'] = df['Age'].fillna(30)
    df['BMI'] = df['BMI'].fillna(22)
    df['Systolic_BP'] = df['Systolic_BP'].fillna(120)
    df['Diastolic_BP'] = df['Diastolic_BP'].fillna(80)

    # Convert Yes/No to 0/1
    df['Exercise'] = df['Exercise'].map({'Yes':0,'No':1}).fillna(0)
    df['Family_History'] = df['Family_History'].map({'Yes':1,'No':0}).fillna(0)
    df['Smoking'] = df['Smoking'].map({'Yes':1,'No':0}).fillna(0)

    # Predict for each row
    results = []
    for _, row in df.iterrows():
        result, risk = predict_extended(
            row['FBS'], row['RBS'], row['HbA1c'],
            row['Age'], row['BMI'],
            row['Systolic_BP'], row['Diastolic_BP'],
            row['Exercise'], row['Family_History'], row['Smoking']
        )

        # Generate QR code
        phone = "+923345534869"
        msg = "I want to order NOORÉ Skin Cream"
        url = f"https://wa.me/{phone[1:]}?text={msg.replace(' ','%20')}"
        qr = qrcode.QRCode(box_size=4)
        qr.add_data(url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = BytesIO()
        img.save(buf, format="PNG")
        qr_base64 = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

        results.append({
            'Blood_Sugar_Status': result,
            'Risk_Factor_Score': risk,
            'QR_Code': qr_base64
        })

    result_df = pd.DataFrame(results)
    # Merge original data with predictions
    final_df = pd.concat([df.reset_index(drop=True), result_df], axis=1)

    # Save CSV for download
    final_df.to_csv("blood_sugar_results.csv", index=False)
    return final_df

# Step 5: Define Gradio Interface
iface = gr.Interface(
    fn=process_csv,
    inputs=gr.File(label="Upload CSV File with Patient Data"),
    outputs=gr.Dataframe(label="Predicted Results"),
    title="🩸 Blood Sugar Batch Predictor",
    description="Upload a CSV file with patient data. Columns: FBS,RBS,HbA1c,Age,BMI,Systolic_BP,Diastolic_BP,Exercise,Family_History,Smoking"
)

# Step 6: Launch app
iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df6da36a1ec4c7dbbe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
